# 01_silver
Goal: turn the raw events (bronze) into one clean, typed table (silver) that the gold tables can trust.

- **Input:** `dev.ecommerce.events_bronze` · 233,460,662 rows · all columns are text
- **Output:** `dev.ecommerce.events_silver` · 233,080,143 rows · typed · clustered by `event_date`
- **Grain:** one action (view / cart / purchase) by one user, on one product, at one moment, in one session
- **Key:** event_time + event_type + product_id + user_id + user_session
- **How to run:** Run all. Safe to run again: the table is overwritten, not appended.

## What changes in silver

| Step | Rule | Rows |
|---|---|---|
| Duplicates | exact copies → delete | 380,454 |
| Key conflicts | same key, different values → keep 1 row per key | 13 |
| user_session | NULL → delete the row (key column) | 52 |
| event_time | text → timestamp | all |
| event_date (new) | date from event_time, for daily and monthly counts | all |
| price | text → decimal(10,2) · 0 → NULL | 495,759 set to NULL |
| brand | NULL → 'unknown' | 29,948,661 |
| category_code | NULL → 'unknown' | 47,496,567 |

**Expected rows:** 233,460,662 − 380,454 − 13 − 52 = **233,080,143**

## Decisions
- **Duplicates → delete.** Exact copies of the same event. Keeping them would count views, carts and sales twice.
- **13 key conflicts → keep one row per key.** Real cart events, only one column value is unclear. 11 of them are on 16-17 Nov, during the known cart tracking problem. Which of the 2 rows is kept is not fixed, but 13 rows out of 233M don't change any result.
- **52 rows with no user_session → delete.** The session is an id and part of the key, not a label. 'unknown' would glue 52 unrelated events into one fake visit. Loss: 0.00002%.
- **Price 0 → NULL, not delete.** The view or cart happened, only the price is missing. 0 would mean "free" and pull averages down. NULL means unknown, and avg skips it. No purchase has price 0.
- **Price as decimal(10,2).** Money needs exact values. double is an approximation.
- **brand / category_code NULL → 'unknown'.** They are labels: the row is real, and 'unknown' keeps it visible as its own group. NULL would disappear in filters like `brand != 'samsung'`.
- **IDs stay string.** product_id, category_id and user_id are ids, not amounts. I only count and join them.
- **Clustered by event_date.** Almost every gold query filters by date, so Databricks can skip files. Liquid clustering instead of partitioning: recommended for new Delta tables, and the column can be changed later.

## 1. Read bronze

In [0]:
from pyspark.sql.functions import col, when, to_timestamp, to_date

bronze_df = spark.read.table("dev.ecommerce.events_bronze")

## 2. Remove duplicates and rows without a key

In [0]:
# one row per key: removes the 380,454 exact copies and the 13 key conflicts
silver_df = bronze_df.dropDuplicates(["event_time", "event_type", "product_id", "user_id", "user_session"])

# 52 rows have no session, and the session is part of the key
silver_df = silver_df.filter(col("user_session").isNotNull())

## 3. Fix types

In [0]:
silver_df = silver_df.withColumns({
    # text "2019-10-01 00:00:00 UTC" -> timestamp, z = time zone
    "event_time" : to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss z"),

    # cast before compare: the zeros are written as "0.00", so == '0' finds nothing
    "price" : when(col("price").cast("double") == 0, None)
              .otherwise(col("price"))
              .cast("decimal(10,2)")
})

# separate step: event_date needs the new timestamp, not the old text
silver_df = silver_df.withColumns({
    "event_date" : to_date(col("event_time"))
})

## 4. Fill missing labels

In [0]:
silver_df = silver_df.withColumns({
    "brand"         : when(col("brand").isNull(), 'unknown').otherwise(col("brand")),
    "category_code" : when(col("category_code").isNull(), 'unknown').otherwise(col("category_code"))
})

silver_df.printSchema()

## 5. Save
`overwrite`: running the notebook twice gives the same table. `clusterBy`: rows with the same dates sit in the same files.

In [0]:
silver_df.write.mode("overwrite").clusterBy("event_date").saveAsTable("dev.ecommerce.events_silver")

## 6. Check
All checks in one query on the saved table, so the data is read only once.

| Check | Expected |
|---|---|
| total_rows | 233,080,143 |
| price_null | 495,759 |
| brand_unknown | 29,948,661 |
| category_unknown | 47,496,567 |
| session_null | 0 |
| empty_text | 0 |
| key_duplicates | 0 |

In [0]:
%sql
SELECT
  count(*)                                   AS total_rows,
  count(*) - count(price)                    AS price_null,
  count_if(brand = 'unknown')                AS brand_unknown,
  count_if(category_code = 'unknown')        AS category_unknown,
  count(*) - count(user_session)             AS session_null,
  count_if(brand = '' OR category_code = '') AS empty_text,
  count(*) - count(DISTINCT event_time, event_type, product_id, user_id, user_session) AS key_duplicates
FROM dev.ecommerce.events_silver